# xkvlora  —  Sec 5.2  augmentation: verbatim x-attn + query-side LoRA

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# XKV-LoRA — LoRA the reader's own W_q/W_k/W_v; cross-attend the holder's verbatim KV

Building on XKV (holder KV kept verbatim, reader cross-attends `[K_self;K_holder]/[V_self;V_holder]`), we now
make the reader's OWN projections trainable via hand-rolled LoRA (implemented directly rather than via PEFT). The reader can reshape
how it *encodes its own context* (W_k/W_v) and how it *looks up* keys (W_q). Holder KV is never touched.

$$\text{ctx}=\mathrm{softmax}\!\left(\tfrac{q'\,[K'_{self};K_{holder}]^\top}{\sqrt d}\right)[V'_{self};V_{holder}],\quad
q'=W_Qh+\Delta_q h,\ K'_{self}=\mathrm{RoPE}(W_Kh+\Delta_k h),\ V'_{self}=W_Vh+\Delta_v h$$

**Arms:** `raw` (0 params) / `qonly` / `kv_only` / `qkv`. **Settings:** `same` then `cross`.
Success = `xkv_shared > floor`, `xkv_ctrl` at floor. Two parity gates: manual==native, and rebuilt-raw==native.

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)
BASE="Qwen/Qwen2.5-1.5B"
HOLDER_X="EleutherAI/gpt-neo-1.3B"    # cross-family holder (2048-dim pool)
CFG=dict(
  lora_R=16, bridge_epochs=4, bridge_lr=2e-4,
  n_train=200, n_eval=200, max_ctx=256, gen_len=8,
  arms=["raw","qonly","kv_only","qkv"], settings=["same","cross"],
  keepbest_slice=32,
)
if os.environ.get("KVL_FAST","0")=="1":
    CFG.update(n_train=24,n_eval=24,bridge_epochs=2,keepbest_slice=12)
    print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_xkvlorafast" if os.environ.get("KVL_FAST","0")=="1" else "_xkvlora"
TAG=os.environ.get("KVL_TAG", _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)

## 1. Load frozen reader (Qwen); grab W_Q/W_K/W_V (+biases), RoPE

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,cache_dir=CACHE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
base=AutoModelForCausalLM.from_pretrained(BASE,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)
ids=tok("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): pkv=base(ids,use_cache=True).past_key_values
if not isinstance(pkv,tuple): pkv=pkv.to_legacy_cache() if hasattr(pkv,"to_legacy_cache") else pkv
N_LAYER=len(pkv); _,N_KV,_,D_HEAD=pkv[0][0].shape
N_Q=base.config.num_attention_heads; GROUP=N_Q//N_KV; KV_DIM=2*N_KV*D_HEAD
KDIM=N_KV*D_HEAD; HID=base.config.hidden_size
layers=base.model.layers
WQ=[layers[l].self_attn.q_proj.weight.detach() for l in range(N_LAYER)]
BQ=[getattr(layers[l].self_attn.q_proj,'bias',None) for l in range(N_LAYER)]
WK=[layers[l].self_attn.k_proj.weight.detach() for l in range(N_LAYER)]
BK=[getattr(layers[l].self_attn.k_proj,'bias',None) for l in range(N_LAYER)]
WV=[layers[l].self_attn.v_proj.weight.detach() for l in range(N_LAYER)]
BV=[getattr(layers[l].self_attn.v_proj,'bias',None) for l in range(N_LAYER)]
QDIM=N_Q*D_HEAD
print(f"L={N_LAYER} n_q={N_Q} n_kv={N_KV} group={GROUP} d_head={D_HEAD} kdim={KDIM} qdim={QDIM} hid={HID}",flush=True)
_THETA=float(getattr(base.config,'rope_theta',1e6))
_INV=1.0/(_THETA**(torch.arange(0,D_HEAD,2,device=DEVICE).float()/D_HEAD))
def rope_at(x,positions):
    # x: (...,T,dh); positions: (T,) absolute positions
    ang=torch.outer(positions.float(),_INV); emb=torch.cat([ang,ang],-1)
    cos=emb.cos(); sin=emb.sin()
    t1,t2=x[...,:D_HEAD//2],x[...,D_HEAD//2:]; rr=torch.cat([-t2,t1],-1)
    return x*cos+rr*sin
def _heads(x,nh):  # (T, nh*dh) -> (nh,T,dh)
    T=x.shape[0]; return x.view(T,nh,D_HEAD).permute(1,0,2)
def _unheads(x):   # (nh,T,dh) -> (T, nh*dh)
    nh,T,dh=x.shape; return x.permute(1,0,2).reshape(T,nh*dh)

## 2. Cache tooling; split-context SQuAD; capture reader hidden (input_layernorm) per layer

In [ ]:
def kv_stack(past):
    outs=[]
    for (k,v) in past:
        B,H,T,dh=k.shape
        outs.append(torch.cat([k.permute(0,2,1,3).reshape(B,T,H*dh),
                               v.permute(0,2,1,3).reshape(B,T,H*dh)],-1))
    return torch.stack(outs,0)
def native_ids(text,max_len,tk=None):
    tk=tk or tok
    return tk([text],return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
@torch.no_grad()
def reader_cache(text,max_len):
    ids=native_ids(text,max_len); last=ids[:,-1:]; body=ids[:,:-1]
    leg=base(body,use_cache=True).past_key_values
    if not isinstance(leg,tuple): leg=leg.to_legacy_cache() if hasattr(leg,"to_legacy_cache") else leg
    return kv_stack(leg), last, body
@torch.no_grad()
def capture_h(body):
    # per-layer input_layernorm OUTPUT over the prefix `body` -> the exact `ln` used to make native K/V/Q.
    Hs={}; hooks=[]
    def mk(l):
        def h(mod,inp,out): Hs[l]=out[0].detach()   # (T,HID)
        return h
    for l in range(N_LAYER): hooks.append(layers[l].input_layernorm.register_forward_hook(mk(l)))
    base(body,use_cache=False)
    for h in hooks: h.remove()
    return Hs   # dict l -> (T,HID)

def make_prompt(p,q): return f"{p}\nQuestion: {q}\nAnswer:"
def mask_answer(p,a):
    i=p.find(a); return None if i<0 else p[:i]+"____"+p[i+len(a):]
def build_qa(n,split):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for ex in ds:
        a=ex["answers"]["text"][0].strip()
        if not a or a not in ex["context"] or len(a.split())>4: continue
        m=mask_answer(ex["context"],a)
        if m is None: continue
        items.append(dict(q=ex["question"].strip(),full=ex["context"].strip(),masked=m.strip(),ans=a))
        if len(items)>=n: break
    return items
train_qa=build_qa(CFG["n_train"],"train"); eval_qa=build_qa(CFG["n_eval"],"validation")
print("train",len(train_qa),"eval",len(eval_qa),flush=True)
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
def hit(t,g): return norm(g) in norm(t) if g else False

## 3. Holder caches — reader own KV (same) and cross-family gpt-neo KV projected to reader dim
Identical to XKV. Holder KV is used VERBATIM (per-token), never adapted.

In [ ]:
@torch.no_grad()
def qwen_holder_kv(text):
    vec,_,_=reader_cache(text,CFG["max_ctx"]); return vec   # (L,1,T,KV_DIM) reader-geometry, POST-RoPE keys

tokX=AutoTokenizer.from_pretrained(HOLDER_X,cache_dir=CACHE)
if tokX.pad_token is None: tokX.pad_token=tokX.eos_token
holderX=AutoModelForCausalLM.from_pretrained(HOLDER_X,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in holderX.parameters(): p.requires_grad_(False)
hx_ids=tokX("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): hxp=holderX(hx_ids,use_cache=True).past_key_values
if not isinstance(hxp,tuple): hxp=hxp.to_legacy_cache() if hasattr(hxp,"to_legacy_cache") else hxp
HX_L=len(hxp); _,HX_H,_,HX_D=hxp[0][0].shape
print(f"holderX layers={HX_L} heads={HX_H} dhead={HX_D} kdim={HX_H*HX_D}",flush=True)
@torch.no_grad()
def holderX_kv(text):
    ids=native_ids(text,CFG["max_ctx"],tokX); body=ids[:,:-1]
    leg=holderX(body,use_cache=True).past_key_values
    if not isinstance(leg,tuple): leg=leg.to_legacy_cache() if hasattr(leg,"to_legacy_cache") else leg
    return kv_stack(leg)   # (HX_L,1,T,2*HX_H*HX_D)
HXKDIM=HX_H*HX_D
layer_map=[min(HX_L-1, round(l*(HX_L-1)/(N_LAYER-1))) for l in range(N_LAYER)]
@torch.no_grad()
def fit_cross_proj(n=40):
    Ks=[[] for _ in range(N_LAYER)]; Vs=[[] for _ in range(N_LAYER)]
    for it in train_qa[:n]:
        hv=holderX_kv(make_prompt(it["full"],it["q"]))
        for l in range(N_LAYER):
            lx=layer_map[l]
            Ks[l].append(hv[lx,0,:,:HXKDIM]); Vs[l].append(hv[lx,0,:,HXKDIM:])
    PK=[]; PV=[]
    for l in range(N_LAYER):
        K=torch.cat(Ks[l],0); V=torch.cat(Vs[l],0)
        def proj(X):
            mu=X.mean(0); Xc=X-mu; U,S,Vh=torch.linalg.svd(Xc,full_matrices=False)
            comp=Vh[:KDIM]; return mu,comp
        PK.append(proj(K)); PV.append(proj(V))
    return PK,PV
PKx,PVx=fit_cross_proj()
print("cross-family projections fit; layer_map sample",layer_map[:6],flush=True)
@torch.no_grad()
def holderX_kv_reader_dim(text):
    hv=holderX_kv(text); T=hv.shape[2]; out=torch.zeros(N_LAYER,1,T,KV_DIM,device=DEVICE)
    for l in range(N_LAYER):
        lx=layer_map[l]; muK,cK=PKx[l]; muV,cV=PVx[l]
        K=(hv[lx,0,:,:HXKDIM]-muK)@cK.T; V=(hv[lx,0,:,HXKDIM:]-muV)@cV.T
        out[l,0,:,:KDIM]=K; out[l,0,:,KDIM:]=V
    return out

## 4. LoRAKQV module + manual reader forward (rebuilt self-KV) with XKV cross-attention

`LoRAKQV` holds per-layer low-rank residuals on q,k,v (rank R). The active `arm` gates which are non-zero.
B init ZERO so the untrained module is exactly identity (parity anchor). The reader's OWN self keys/values
are REBUILT from captured hidden `h` with the delta; holder KV is used verbatim.

In [ ]:
class LoRAKQV(nn.Module):
    def __init__(s,arm):
        super().__init__(); s.arm=arm; R=CFG["lora_R"]
        s.use_q = arm in ("qonly","qkv")
        s.use_k = arm in ("kv_only","qkv")
        s.use_v = arm in ("kv_only","qkv")
        def mk(out):
            A=nn.ParameterList([nn.Parameter(0.01*torch.randn(HID,R)) for _ in range(N_LAYER)])
            B=nn.ParameterList([nn.Parameter(torch.zeros(R,out)) for _ in range(N_LAYER)])  # zero -> identity
            return A,B
        if s.use_q: s.Aq,s.Bq=mk(QDIM)
        if s.use_k: s.Ak,s.Bk=mk(KDIM)
        if s.use_v: s.Av,s.Bv=mk(KDIM)
    def dq(s,l,h): return (h@s.Aq[l])@s.Bq[l] if s.use_q else 0.0   # h:(T,HID)->(T,QDIM)
    def dk(s,l,h): return (h@s.Ak[l])@s.Bk[l] if s.use_k else 0.0   # ->(T,KDIM)
    def dv(s,l,h): return (h@s.Av[l])@s.Bv[l] if s.use_v else 0.0

SCALE=1.0/(D_HEAD**0.5)
# Self K,V are REBUILT here from hidden h (so the k/v LoRA touches the whole context). Keys RoPE'd ONCE at
# their own positions (we build them pre-RoPE, unlike XKV which read post-RoPE cache). Holder K post-RoPE
# verbatim (do NOT re-RoPE). V never RoPE'd.
def build_self_kv(lora, l, h_prefix, positions):
    # h_prefix:(T,HID) captured input_layernorm output; positions:(T,) absolute -> K'(T,KDIM) post-RoPE, V'(T,KDIM)
    kflat=h_prefix@WK[l].T + (BK[l] if BK[l] is not None else 0.0) + lora.dk(l,h_prefix)
    vflat=h_prefix@WV[l].T + (BV[l] if BV[l] is not None else 0.0) + lora.dv(l,h_prefix)
    kh=_heads(kflat,N_KV); kh=rope_at(kh,positions); kflat=_unheads(kh)   # post-RoPE
    return torch.cat([kflat,vflat],-1)   # (T,KV_DIM)

def xkv_attend(lora, h_layer_in, l, self_KV, holder_KV, pos_query, T_holder):
    # h_layer_in:(1,1,HID). self_KV:(T_self,KV_DIM) K post-RoPE, V raw (already rebuilt w/ delta incl. current
    # token). holder_KV:(T_holder,KV_DIM) K post-RoPE verbatim, V raw. returns attn output (1,1,HID).
    ln=layers[l].input_layernorm(h_layer_in).reshape(1,HID)
    qflat=ln@WQ[l].T + (BQ[l] if BQ[l] is not None else 0.0) + lora.dq(l,ln)   # (1,QDIM)
    q=_heads(qflat,N_Q); q=rope_at(q, torch.tensor([pos_query],device=DEVICE))
    Ks=_heads(self_KV[:,:KDIM],N_KV); Vs=_heads(self_KV[:,KDIM:],N_KV)
    Ks_e=Ks.repeat_interleave(GROUP,0); Vs_e=Vs.repeat_interleave(GROUP,0)
    if T_holder>0:
        Kh=_heads(holder_KV[:,:KDIM],N_KV); Vh=_heads(holder_KV[:,KDIM:],N_KV)
        Kall=torch.cat([Ks_e,Kh.repeat_interleave(GROUP,0)],1)
        Vall=torch.cat([Vs_e,Vh.repeat_interleave(GROUP,0)],1)
    else:
        Kall=Ks_e; Vall=Vs_e
    att=torch.einsum('hqd,hkd->hqk', q, Kall)*SCALE
    w=torch.softmax(att,dim=-1)
    ctx=torch.einsum('hqk,hkd->hqd', w, Vall).permute(1,0,2).reshape(1,QDIM)
    return layers[l].self_attn.o_proj(ctx).view(1,1,HID)

def reader_block_forward(l, h_in, attn_out):
    h=h_in+attn_out; ln2=layers[l].post_attention_layernorm(h); return h+layers[l].mlp(ln2)

## 4b. Full XKV-LoRA greedy generation (rebuilds self prefix KV with deltas, grows own cache)

In [ ]:
def xkv_generate(lora, setting, Hs_prefix, last_id, holder_vec):
    # Hs_prefix: dict l->(T_self,HID) captured masked-prefix hidden. holder_vec:(L,1,T_h,KV_DIM).
    T_self=Hs_prefix[0].shape[0]; T_h=holder_vec.shape[2]
    pos_prefix=torch.arange(T_self,device=DEVICE)
    selfKV=[build_self_kv(lora,l,Hs_prefix[l],pos_prefix) for l in range(N_LAYER)]  # each (T_self,KV_DIM)
    cur=last_id; gen=[]; pos=T_self
    for step in range(CFG["gen_len"]):
        h=base.model.embed_tokens(cur)   # (1,1,HID)
        for l in range(N_LAYER):
            ln=layers[l].input_layernorm(h).reshape(1,HID)
            k=ln@WK[l].T + (BK[l] if BK[l] is not None else 0.0) + lora.dk(l,ln)
            v=ln@WV[l].T + (BV[l] if BV[l] is not None else 0.0) + lora.dv(l,ln)
            kh=_heads(k,N_KV); kh=rope_at(kh, torch.tensor([pos],device=DEVICE)); k=_unheads(kh)  # post-RoPE
            selfKV[l]=torch.cat([selfKV[l], torch.cat([k,v],-1)],0)
            attn=xkv_attend(lora,h,l,selfKV[l],holder_vec[l,0],pos,T_h)
            h=reader_block_forward(l,h,attn)
        h=base.model.norm(h); logit=base.lm_head(h)[:, -1, :]
        nxt=logit.argmax(-1,keepdim=True); gen.append(nxt); cur=nxt; pos+=1
    return tok.decode(torch.cat(gen,1)[0])

## 4c. PARITY GATES — manual==native AND rebuilt-raw prefix == native cache

Gate 1: `xkv_generate` with a raw LoRA and ZERO-length holder must == native greedy (hand-rolled attention
sound). Gate 2: rebuilding the prefix K/V from captured h with raw (delta=0) must match the native cache to
~1e-4 (the capture/RoPE reconstruction is faithful). Both must pass before any transfer number is trusted.

In [ ]:
from transformers.cache_utils import DynamicCache
def slots_to_legacy(vec):
    L,B,Kk,_=vec.shape; half=KDIM; leg=[]
    for l in range(L):
        kt=vec[l,:,:,:half].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        vt=vec[l,:,:,half:].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        leg.append((kt,vt))
    return leg
@torch.no_grad()
def native_greedy(prefix_vec,last_id):
    past=DynamicCache.from_legacy_cache(tuple(slots_to_legacy(prefix_vec))); T0=prefix_vec.shape[2]
    def pos(s,l): return torch.arange(s,s+l,device=DEVICE).unsqueeze(0)
    o=base(last_id,past_key_values=past,position_ids=pos(T0,1),
           attention_mask=torch.ones(1,T0+1,device=DEVICE),use_cache=True)
    past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen=[cur]; L=T0+1
    for _ in range(CFG["gen_len"]-1):
        o=base(cur,past_key_values=past,position_ids=pos(L,1),
               attention_mask=torch.ones(1,L+1,device=DEVICE),use_cache=True)
        past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen.append(cur); L+=1
    return tok.decode(torch.cat(gen,1)[0])

# Gate 2: rebuilt prefix (raw) vs native cache
_raw=LoRAKQV("raw").to(DEVICE); _mx=0.0
for it in eval_qa[:min(4,len(eval_qa))]:
    vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
    Hs=capture_h(body); T=vec.shape[2]; posp=torch.arange(T,device=DEVICE)
    for l in range(N_LAYER):
        reb=build_self_kv(_raw,l,Hs[l],posp)            # (T,KV_DIM)
        _mx=max(_mx, float((reb-vec[l,0]).abs().max()))
print(f"REBUILT-RAW vs native cache maxabs diff: {_mx:.3e}",flush=True)
assert _mx<1e-2, "rebuilt prefix K/V diverges from native cache — capture/RoPE path wrong"

# Gate 1: manual==native
_match=0; _tot=0
for it in eval_qa[:min(8,len(eval_qa))]:
    vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
    Hs=capture_h(body)
    man=xkv_generate(_raw,"same",Hs,last,vec[:, :, :0, :])
    nat=native_greedy(vec,last)
    _tot+=1; _match+= (man.strip()==nat.strip())
    if _tot<=3: print(f"  manual={man!r}\n  native={nat!r}\n  ---",flush=True)
print(f"PARITY manual==native: {_match}/{_tot}",flush=True)
assert _match>=_tot-1, "manual forward diverges from native — floor/ceiling would be invalid"
print("parity OK",flush=True)

## 5. Train LoRA on answer CE (decision step) with KEEP-BEST guard vs raw

In [ ]:
def holder_for(setting, it, masked=False):
    text=make_prompt(it["masked"] if masked else it["full"], it["q"])
    return qwen_holder_kv(text) if setting=="same" else holderX_kv_reader_dim(text)

def ce_loss_xkv(lora, setting, it):
    # decision step: rebuild masked-prefix self KV (with delta), attend over [self;holder-full], predict gold[0]
    vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
    Hs=capture_h(body)
    holder_vec=holder_for(setting,it,masked=False)
    gold=tok(" "+it["ans"],return_tensors="pt").input_ids.to(DEVICE)[:, :1] if it["ans"] else None
    if gold is None: return None
    T_self=Hs[0].shape[0]; T_h=holder_vec.shape[2]; pos=T_self
    posp=torch.arange(T_self,device=DEVICE)
    selfKV=[build_self_kv(lora,l,Hs[l],posp) for l in range(N_LAYER)]
    h=base.model.embed_tokens(last)
    for l in range(N_LAYER):
        attn=xkv_attend(lora,h,l,selfKV[l],holder_vec[l,0],pos,T_h)
        h=reader_block_forward(l,h,attn)
    h=base.model.norm(h); logit=base.lm_head(h)[:, -1, :]
    return F.cross_entropy(logit, gold.view(-1))

@torch.no_grad()
def eval_shared_slice(lora,setting,items):
    # quick xkv_shared on a slice, for keep-best
    n=0; hits=0
    for it in items:
        vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
        Hs=capture_h(body); hv=holder_for(setting,it,masked=False)
        hits+=hit(xkv_generate(lora,setting,Hs,last,hv),it["ans"]); n+=1
    return hits/max(1,n)

def train_lora(arm,setting,data):
    lora=LoRAKQV(arm).to(DEVICE)
    if arm=="raw": return lora     # nothing to train
    slice_items=data[:CFG["keepbest_slice"]]
    # epoch-0 sanity: zero-init => equals raw on the slice
    raw0=eval_shared_slice(LoRAKQV("raw").to(DEVICE),setting,slice_items)
    ep0=eval_shared_slice(lora,setting,slice_items)
    assert abs(ep0-raw0)<1e-9, f"{arm}:{setting} untrained != raw (raw0={raw0} ep0={ep0}) — residual init wrong"
    best_state={k:v.detach().clone() for k,v in lora.state_dict().items()}; best=max(raw0,ep0)
    print(f"  [{setting}:{arm}] ep0 slice_shared={ep0:.3f} (raw={raw0:.3f}) best={best:.3f}",flush=True)
    opt=torch.optim.Adam([p for p in lora.parameters()],lr=CFG["bridge_lr"])
    for ep in range(CFG["bridge_epochs"]):
        idx=torch.randperm(len(data)).tolist(); tot=0.0; n=0
        for j in idx:
            L=ce_loss_xkv(lora,setting,data[j])
            if L is None or not torch.isfinite(L): continue
            opt.zero_grad(); L.backward(); torch.nn.utils.clip_grad_norm_(lora.parameters(),1.0); opt.step()
            tot+=float(L); n+=1
        sh=eval_shared_slice(lora,setting,slice_items)
        keep=""
        if sh>best: best=sh; best_state={k:v.detach().clone() for k,v in lora.state_dict().items()}; keep=" <-keep"
        print(f"  [{setting}:{arm}] ep{ep+1} ce={tot/max(1,n):.4f} slice_shared={sh:.3f} best={best:.3f}{keep}",flush=True)
    lora.load_state_dict(best_state)   # never worse than the best seen (>= raw by construction)
    return lora

## 6. Eval: floor / ceiling / identity(same) / xkv_shared / xkv_ctrl

In [ ]:
@torch.no_grad()
def eval_xkv(lora,setting,items):
    A={"floor":0,"ceiling":0,"identity":0,"xkv_shared":0,"xkv_ctrl":0}; n=0
    raw=LoRAKQV("raw").to(DEVICE)
    for it in items:
        g=it["ans"]
        vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"]); Hs=capture_h(body)
        empty=vec[:, :, :0, :]
        A["floor"]+=hit(xkv_generate(raw,setting,Hs,last,empty),g)                 # masked self only (raw)
        fvec,lastf,fbody=reader_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"]); Hf=capture_h(fbody)
        A["ceiling"]+=hit(xkv_generate(raw,setting,Hf,lastf,empty),g)              # full context (raw)
        if setting=="same":
            hv=qwen_holder_kv(make_prompt(it["full"],it["q"]))
            A["identity"]+=hit(xkv_generate(raw,setting,Hs,last,hv),g)             # masked self + own-full holder, raw
        hv=holder_for(setting,it,masked=False)
        A["xkv_shared"]+=hit(xkv_generate(lora,setting,Hs,last,hv),g)              # masked self + holder full, trained
        hvc=holder_for(setting,it,masked=True)
        A["xkv_ctrl"]+=hit(xkv_generate(lora,setting,Hs,last,hvc),g)              # neg control: holder masked
        n+=1
    return {k:v/max(1,n) for k,v in A.items()}

## 7. Run arms x settings + save + verdict

In [ ]:
RES={"config":CFG,"base":BASE,"holderX":HOLDER_X,"runs":{}}
RESPATH=f"results/xkvlora{TAG}.json"
for setting in CFG["settings"]:
    for arm in CFG["arms"]:
        print(f"\n########## {setting} : {arm} ##########",flush=True)
        lora=train_lora(arm,setting,train_qa)
        m=eval_xkv(lora,setting,eval_qa)
        RES["runs"][f"{setting}:{arm}"]=m
        ids=f" identity={m['identity']:.3f}" if setting=="same" else ""
        print(f"[{setting}:{arm}] floor={m['floor']:.3f} ceil={m['ceiling']:.3f}{ids} "
              f"xkv_shared={m['xkv_shared']:.3f} xkv_ctrl={m['xkv_ctrl']:.3f}",flush=True)
        json.dump(RES,open(RESPATH,"w"),indent=1)

print("\n=== XKV-LoRA VERDICT: does LoRA-ing reader q/k/v help read the holder's real KV? ===",flush=True)
for setting in CFG["settings"]:
    fl=RES["runs"][f"{setting}:raw"]["floor"]; ce=RES["runs"][f"{setting}:raw"]["ceiling"]
    print(f" [{setting}] floor={fl:.3f} ceiling={ce:.3f}"
          +(f" identity={RES['runs'][f'{setting}:raw']['identity']:.3f}" if setting=='same' else ""),flush=True)
    for arm in CFG["arms"]:
        m=RES["runs"][f"{setting}:{arm}"]
        beat=" <-- BEATS FLOOR" if m["xkv_shared"]>fl+1e-9 else ""
        cln=" (ctrl clean)" if m["xkv_ctrl"]<=fl+0.03 else " (ctrl LIFTS - suspect)"
        print(f"   {arm:8s} shared={m['xkv_shared']:.3f} ctrl={m['xkv_ctrl']:.3f}{cln}{beat}",flush=True)
json.dump(RES,open(RESPATH,"w"),indent=1); print("saved",RESPATH,flush=True)

## 8. Plot

In [ ]:
fig,ax=plt.subplots(1,len(CFG["settings"]),figsize=(6.6*len(CFG["settings"]),4.4),squeeze=False)
for si,setting in enumerate(CFG["settings"]):
    a=ax[0][si]; fl=RES["runs"][f"{setting}:raw"]["floor"]; ce=RES["runs"][f"{setting}:raw"]["ceiling"]
    xs=CFG["arms"]
    sh=[RES["runs"][f"{setting}:{k}"]["xkv_shared"] for k in xs]
    ct=[RES["runs"][f"{setting}:{k}"]["xkv_ctrl"] for k in xs]
    x=np.arange(len(xs))
    a.bar(x-0.2,sh,0.4,label="xkv_shared",color="#2c7fb8")
    a.bar(x+0.2,ct,0.4,label="xkv_ctrl",color="#d95f0e")
    a.axhline(fl,ls='--',color='#999',label=f"floor {fl:.2f}"); a.axhline(ce,ls='-',color='#000',label=f"ceil {ce:.2f}")
    a.set_xticks(x); a.set_xticklabels(xs); a.set_title(f"XKV-LoRA — {setting}"); a.legend(fontsize=8); a.grid(ls=':',alpha=.4)
fig.tight_layout(); fig.savefig("figures/xkvlora_curves.png",dpi=150); plt.show()